<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# IMU Rates and Configuration

A physical Duckiedrone publishes `/mavros/imu/data` at about $50\,\mathrm{Hz}$, but nothing in the hardware forces that number. It is a choice, made at one place in the path that [Notebook 5](./5-duckiedrone-imu-architecture.ipynb) followed. This notebook finds that place, changes the rate of a live stream from a terminal, and shows two settings that look like topic rates but are not.

By the end of the notebook you should be able to answer:

- Which setting decides how often each IMU topic is published?

- Why does changing a PX4 IMU parameter leave the topic rate unchanged?

- How does a stored parameter differ from a stream-rate request?

## Configuring IMU rates and streams

Three rates in [Table 1](./5-duckiedrone-imu-architecture.ipynb#table-1) are configurable: $800\,\mathrm{Hz}$ set by `IMU_GYRO_RATEMAX`, $250\,\mathrm{Hz}$ set by `IMU_INTEG_RATE`, and the $50\,\mathrm{Hz}$ MAVLink stream. Suppose an application needs an IMU topic at a different rate. Which number should change?

`IMU_INTEG_RATE` controls one upstream rate. Changing it alone does not change either ROS 2 topic rate, because those rates are set at the MAVLink boundary. [Figure 1](./5-duckiedrone-imu-architecture.ipynb#figure-1) shows why these settings affect different parts of the pipeline.

### Settings the flight controller remembers

`IMU_INTEG_RATE` and `IMU_GYRO_RATEMAX` are PX4 parameters: values stored on the flight controller that survive a reboot. A new value of either one takes effect only after the flight controller reboots. The [PX4 parameter reference](https://docs.px4.io/main/en/advanced_config/parameter_reference.html) describes both. They change the data PX4 prepares, not how often it is sent to the Raspberry Pi.

### What actually crosses the link

Sending is a separate decision. When the link to the Raspberry Pi comes up, PX4 configures which MAVLink messages it will send and how often to send each one. The IMU uses two of them: `HIGHRES_IMU` carries the accelerometer and gyroscope, and `ATTITUDE_QUATERNION` carries the fused orientation and angular rate.

Those messages feed different ROS 2 topics. MAVROS2 publishes `/mavros/imu/data_raw` when `HIGHRES_IMU` arrives. It publishes `/mavros/imu/data` when `ATTITUDE_QUATERNION` arrives, combining its orientation and angular rate with the latest acceleration from `HIGHRES_IMU`. If the two messages run at different rates, a `data` message can therefore contain a new attitude and a previously received acceleration.

Each MAVLink message interval directly controls its corresponding ROS 2 output: `data_raw` follows `HIGHRES_IMU`, while `data` follows `ATTITUDE_QUATERNION`. To change a particular topic rate, adjust its source-message interval rather than an unrelated upstream parameter. [MAVLink Messaging](https://docs.px4.io/main/en/middleware/mavlink.html) describes how PX4 configures these streams.

### Asking for a different rate while everything is running

A reboot is not needed to request a different stream rate. The Raspberry Pi can ask the flight controller for one while it is running, using the MAVLink `SET_MESSAGE_INTERVAL` command (number 511). The command takes the message id and the interval in $\mu\mathrm{s}$. MAVROS2 exposes that command as a ROS 2 service, so a terminal is enough. Send it only while the Duckiedrone is disarmed.

If the Duckiedrone publishes under a namespace, replace `/mavros` in the service and topic paths below with `/DUCKIEDRONE_NAME/mavros`.

`HIGHRES_IMU` is message id 105, and $100000\,\mu\mathrm{s}$ is $0.1\,\mathrm{s}$, so this requests $10\,\mathrm{Hz}$ for `/mavros/imu/data_raw`:

```bash
ros2 service call /mavros/cmd/command mavros_msgs/srv/CommandLong \
  "{command: 511, param1: 105.0, param2: 100000.0}"
```

Leave `ros2 topic hz /mavros/imu/data_raw` running in another terminal while sending it. The measured rate should approach $10\,\mathrm{Hz}$; timing on a virtual Duckiedrone can introduce variation. Send `20000.0` to request $50\,\mathrm{Hz}$ again. A shorter interval asks for a faster stream; whether PX4 and the link keep up with it has to be measured with the same `ros2 topic hz` command. This request does not set `/mavros/imu/data` to $10\,\mathrm{Hz}$: that topic is published by `ATTITUDE_QUATERNION`, whose interval must be requested separately when an application needs both topics at the same rate.

Unlike a parameter, this request is not persistent. Reboot the flight controller and the rate returns to the link's configured value. The MAVROS2 launcher uses the same command for message id 132, `DISTANCE_SENSOR`, with an interval of `-1` to disable that stream when it is not needed. The request must be sent again on every launch. The message ids, and the meaning of each command parameter, are in the [MAVLink common message set](https://mavlink.io/en/messages/common.html).

### One switch that looks like a rate and is not

One more setting sits on the Raspberry Pi. MAVROS2 loads a list of plugins from `px4_config.yaml`, and a plugin turns one family of MAVLink messages into ROS 2 topics. The `imu` plugin creates `/mavros/imu/data` at root, or `/DUCKIEDRONE_NAME/mavros/imu/data` when MAVROS2 uses that namespace.

Removing that plugin removes its ROS 2 topics, but PX4 continues sending the corresponding MAVLink messages at the configured rate. Disabling the plugin stops ROS 2 from consuming the messages; it does not reduce traffic on the link. To reduce that traffic, change the corresponding MAVLink message interval.

## Further reading

[Finding/Updating Parameters](https://docs.px4.io/main/en/advanced_config/parameters.html) shows how to read and change PX4 parameters in QGroundControl.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()